In [ ]:
# ---------------------------------------------------------------------------
# Batch stack concatenator
# ---------------------------------------------------------------------------
# During segmentation, volumes are processed in smaller batches resulting in multiple separate 3D
# TIFF stacks. This script reassembles those batches into a single continuous
# sequence by reading each stack in order and saving every slice as an
# individually numbered 2D TIFF file (e.g. slice_0000.tif, slice_0001.tif, …).
# The zero-padded naming ensures that the original slice order is preserved
# across batch boundaries, producing a complete, correctly ordered dataset
# ready for downstream analysis.
# ---------------------------------------------------------------------------

import os
import numpy as np
import imageio

# ---------------------------------------------------------------------------
# Configuration — replace these values before running
# ---------------------------------------------------------------------------
# List of absolute paths to the multi-slice TIFF stacks to be processed.
# Each entry should point to a single 3D TIFF file (Z, Y, X).
# Add or remove entries to match the number of stacks in your dataset.
input_files = [
    "/path/to/stack_part1.tif",
    "/path/to/stack_part2.tif",
    "/path/to/stack_part3.tif",
]

# Absolute path to the folder where individual 2D slice TIFFs will be saved.
# The folder will be created automatically if it does not already exist.
output_folder = "/path/to/output/folder"

# ---------------------------------------------------------------------------
# Output directory setup
# ---------------------------------------------------------------------------
# exist_ok=True prevents an error if the folder already exists, making the
# script safe to re-run without manual cleanup.
os.makedirs(output_folder, exist_ok=True)

# ---------------------------------------------------------------------------
# Stack reading and slice extraction
# ---------------------------------------------------------------------------
# global_index tracks the slice number across all stacks so that output
# filenames are unique and sequentially ordered even when multiple stacks
# are concatenated (e.g., slice_0000.tif, slice_0001.tif, …).
global_index = 0

for file_path in input_files:

    print(f"Reading stack from {file_path} ...")

    # imageio.volread loads a multi-page TIFF as a 3D NumPy array with
    # shape (Z, Y, X), where Z is the number of slices (pages) and
    # Y, X are the spatial dimensions of each slice.
    volume = imageio.volread(file_path)

    print(f"Stack shape: {volume.shape} (slices, height, width)")

    for i in range(volume.shape[0]):

        # Extract a single 2D slice along the Z axis.
        slice_2d = volume[i]

        # Zero-padded filename ensures correct alphabetical sorting
        # when slices are later loaded as a sequence (e.g., slice_0000.tif).
        out_filename = f"slice_{global_index:04d}.tif"

        imageio.imwrite(os.path.join(output_folder, out_filename), slice_2d)

        global_index += 1

print(f"Done: {global_index} slices saved as individual TIFFs in '{output_folder}'.")